# Analyze and save Size Exclusion Chromatography (SEC) spectra

In [ ]:
import sys
from pathlib import Path

import pandas as pd

%load_ext autoreload
%autoreload 2

REPO_ROOT = Path.cwd().parent

SEC_ANALYSIS = REPO_ROOT / "SEC_analysis"

if SEC_ANALYSIS not in sys.path:
    sys.path.insert(0, str(SEC_ANALYSIS))

In [ ]:
from plot_sec_to_svg import get_series_pairs, render_sec_svg
from sec_raw_to_normalized import run_all_proteins

SEC_DATA_DIR = Path("data") / "SEC"
DATA_DIR = SEC_ANALYSIS / "normalized"
OUTPUT_DIR = Path("plots") / "SEC"


DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run_all_proteins(SEC_DATA_DIR, DATA_DIR, volume_min=0.1, volume_max=25.0)


In [ ]:
xlsx_files = sorted(DATA_DIR.glob("*.xlsx"))
xlsx_files

**Batch run:** Loop over all `.xlsx` files in `data/` and write one SVG per file (same behavior as running `python plot_sec_to_svg.py` when the script is configured to use `data/` and `output/`).


In [ ]:
# X-axis limit override to avoid artifacts (right edge, mL).
XLIM_RIGHT = {
    "bsa_curves_normalized": 22,
    "DNAse_curves_normalized": 23.5,
}

for excel_path in xlsx_files:
    xl = pd.ExcelFile(excel_path)
    if "Data" not in xl.sheet_names:
        print(f"  Skip {excel_path.name}: no 'Data' sheet")
        continue
    df = pd.read_excel(excel_path, sheet_name="Data", header=0)
    pairs = get_series_pairs(df)
    if not pairs:
        print(f"  Skip {excel_path.name}: no series found")
        continue
    title = excel_path.stem.replace("_", " ").replace(" (2)", "").replace(" (27)", "")
    f = render_sec_svg(pairs, legend_title=title)
    x_max = XLIM_RIGHT.get(excel_path.stem)
    if x_max is not None:
        f.axes[0].set_xlim(right=x_max)
    f.savefig(OUTPUT_DIR / f"{excel_path.stem}.svg", format="svg", bbox_inches="tight")
